# Mount Elgon flood records — preprocessing

Cleans the DesInventar disaster inventory for the nine Mount Elgon districts,
augmented with EM-DAT records, into a **district-day** frame suitable for training
a flood prediction model, where one row answers "was district D flooded on day T".

**Sources:**

- `dataset/mountelgon-desinventar.xls` — DesInventar, 271 rows, 9 districts,
  1933–2018
- `dataset/emdat-uganda.xlsx` — EM-DAT, 39 Ugandan disaster rows, of which 17
  name a Mount Elgon district; extends coverage to 2025

The notebook works on a single `df`, reassigned as each step is applied:

| Step | Effect on `df` |
| --- | --- |
| 1. Load DesInventar | 271 rows |
| 2. Merge EM-DAT records | 271 → 320 |
| 3. Drop invalid dates | 320 → 272 |
| 4. Collapse to one row per district-day | 272 → 145 |
| 5. Group into multi-district storms | (no row change) |
| 6. Expand to day spans (21d DesInventar / 2d EM-DAT) | 145 → 279 |

> Cells are order-dependent — run top to bottom. Re-running a step out of
> sequence will report against an already-cleaned frame.

## 1. Setup and load

`dataset_path` is relative to the repository root. A notebook has no `__file__`
and the kernel's working directory depends on where Jupyter was launched, so the
root is found by walking up to the directory containing `dataset/` rather than by
a fixed number of `.parents` hops.

In [203]:
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

dataset_path = "dataset/mountelgon-desinventar.xls"

DATE_COLUMN = "Date (YMD)"


pd.set_option('display.max_rows', None) # No cap on rows printed out


def find_repo_root(marker: str = "dataset") -> Path:
    """Walk up from the working directory until `marker` is found."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"No parent of {Path.cwd()} contains {marker!r}")


# `dataset_path` is relative to the repository root
REPO_ROOT = find_repo_root()

path = REPO_ROOT / dataset_path
if not path.exists():
    raise FileNotFoundError(f"Dataset not found: {path}")

df = pd.read_excel(path)
print(f"loaded {len(df)} rows x {len(df.columns)} columns from {path.name}")

loaded 271 rows x 17 columns from mountelgon-desinventar.xls


### What the columns hold

Locations are recorded at three nested levels — **District > Subcounty >
Parish** — with a free-text `Location` used when `Subcounty` and `Parish` are
blank. This matters for the duplicate check in section 3: two rows sharing a
date and district are usually *different sub-locations*, not repeats.

Two columns are worth noting before any cleaning:

- **`Event` is `FLOOD` in all 271 rows**, so it carries no information and any
  "same event" grouping below reduces to a date + place comparison.
- **`Deaths` is recorded in only 17 rows** (87 deaths total) and
  `Duration (d)` in 57, so most rows describe an event without quantifying it.

In [204]:
print(df[["Date (YMD)", "Duration (d)", "District", "Subcounty", "Parish", "Location"]].head(10))

print("\nnon-null counts for the key fields:")
print(df[["Event", "Cause", "District", "Subcounty", "Parish", "Location", "Deaths", "Duration (d)"]].notna().sum().to_string())

print(f"\ndistinct Event values: {df['Event'].unique().tolist()}")
print(f"rows per district:\n{df['District'].value_counts().to_string()}")

   Date (YMD)  Duration (d)   District  Subcounty Parish Location
0    1933/0/0           NaN     BUDUDA        NaN    NaN      NaN
1    1933/0/0           NaN     BUDUDA        NaN    NaN      NaN
2  2014/10/17           NaN  BULAMBULI     SISIYI    NaN      NaN
3    2018/6/7           NaN  BULAMBULI        NaN    NaN      NaN
4    2018/5/1           NaN   BUTALEJA        NaN    NaN      NaN
5    2014/6/4           NaN    SIRONKO      ZESUI    NaN      NaN
6    1991/9/2           NaN      MBALE    BUFUMBO    NaN      NaN
7  2013/10/25           NaN    SIRONKO     BUYOBO    NaN      NaN
8    2012/7/3           NaN     BUDUDA  BULUCHEKE    NaN      NaN
9    2002/6/1           NaN    SIRONKO   BUMASIFA    NaN      NaN

non-null counts for the key fields:
Event           271
Cause           228
District        271
Subcounty       149
Parish           43
Location         78
Deaths           17
Duration (d)     57

distinct Event values: ['FLOOD']
rows per district:
District
BUTALEJA     68

## 2. Adding EM-DAT records

DesInventar is not the only inventory covering these districts, and it has a hard
limitation: **it stops in 2018.** EM-DAT (the Emergency Events Database,
maintained by CRED at UCLouvain) records Ugandan floods through 2025, so merging
it extends the record by seven years — including the major 2019, 2022 and 2025
events that DesInventar has no rows for at all.

The two inventories are built differently, and that shapes how they can be joined:

| | DesInventar | EM-DAT |
| --- | --- | --- |
| Unit of record | one district (often one sub-county) | one national disaster event |
| Geography | `District` / `Subcounty` / `Parish` columns | districts named in free-text `Location` and JSON `Admin Units` |
| Dates | single `Date (YMD)` + `Duration (d)` | `Start Year/Month/Day` and `End Year/Month/Day` |
| Impact figures | per district | per event, aggregated across all districts |

### How EM-DAT rows are matched and reshaped

1. **District detection.** Each EM-DAT row is scanned for any of the nine Mount
   Elgon district names, case-insensitively and on whole words. The names appear
   in `Location` prose and inside the JSON of `Admin Units` / `GADM Admin Units`,
   so every cell is searched rather than one designated column.
2. **Explosion to district level.** One EM-DAT event naming seven districts
   becomes seven rows, because a DesInventar row means "this district, this day"
   and the frame must stay at that resolution.
3. **Duration.** `end - start + 1`, so a single-day flood is 1 day rather than 0.
   DesInventar defines `Duration (d)` as how long the event lasted, and both
   sources must mean the same thing by it.
4. **Provenance.** A new `Source Dataset` column marks every row as
   `DesInventar` or `EM-DAT`, and `Source` carries the EM-DAT event id (e.g.
   `EM-DAT 2019-0625-UGA`) so any row traces back to its source record.

### What is deliberately not carried over

**EM-DAT's `Total Deaths` and `Total Affected` are event-level totals spanning
every district at once.** Record `2019-0625-UGA` reports 65 deaths across 7
matched districts; copying that figure onto each district row would report 455
deaths, and dividing it would invent a per-district split the source does not
support. Both columns are therefore omitted, and EM-DAT rows carry a null
`Deaths`. These rows contribute **flood occurrence and timing only** — which is
what a district-day prediction target actually needs.

Likewise `Subcounty`, `Parish` and `Location` stay null: EM-DAT does not record
below district level, so these rows are inherently coarser than DesInventar's.

### A note on the district list

The district names are spelled as EM-DAT spells them. In particular the district
is **Kapchorwa**; an earlier version of the extraction script searched for
"Kapchopwa", which matched nothing and silently dropped 6 records naming that
district.

In [205]:
EMDAT_PATH = REPO_ROOT / "dataset/emdat-uganda.xlsx"

# Spelled as EM-DAT spells them - note "Kapchorwa", not the "Kapchopwa" variant
MOUNT_ELGON_DISTRICTS = [
    "BUTALEJA", "MBALE", "MANAFWA", "BUDUDA", "SIRONKO",
    "BUKWO", "KWEEN", "KAPCHORWA", "BULAMBULI",
]

# Well clear of DesInventar's own Serial range, so provenance is obvious on sight
EMDAT_SERIAL_BASE = 900_001

EMDAT_DATE_PARTS = ["Start Year", "Start Month", "Start Day", "End Year", "End Month", "End Day"]


def districts_mentioned(row: pd.Series, patterns: dict[str, re.Pattern]) -> list[str]:
    """Every Mount Elgon district named anywhere in one EM-DAT row.

    Every cell is searched, not just `Location`: the district names also appear
    inside the JSON of the `Admin Units` and `GADM Admin Units` columns, which is
    where most matches actually come from. Whole-word matching avoids a district
    name matching inside an unrelated longer word.
    """
    return [
        district
        for district, pattern in patterns.items()
        if any(pattern.search(str(value)) for value in row if pd.notna(value))
    ]


def extract_emdat_records(path: Path, districts: list[str]) -> tuple[pd.DataFrame, dict[str, int]]:
    """EM-DAT flood rows reshaped to the DesInventar schema, one row per district."""
    emdat = pd.read_excel(path)
    patterns = {
        district: re.compile(rf"\b{re.escape(district)}\b", re.IGNORECASE)
        for district in districts
    }

    records, matched_events, skipped = [], 0, []
    for _, row in emdat.iterrows():
        matched = districts_mentioned(row, patterns)
        if not matched:
            continue
        # A partial start/end date cannot give a duration, so such rows are
        # reported rather than silently guessed at
        if row[EMDAT_DATE_PARTS].isna().any():
            skipped.append(row["DisNo."])
            continue

        matched_events += 1
        start = datetime(int(row["Start Year"]), int(row["Start Month"]), int(row["Start Day"]))
        end = datetime(int(row["End Year"]), int(row["End Month"]), int(row["End Day"]))

        for district in matched:
            records.append(
                {
                    "Event": "FLOOD",
                    # Unpadded, matching DesInventar's own "2018/6/7" style
                    DATE_COLUMN: f"{start.year}/{start.month}/{start.day}",
                    # Inclusive: a same-day flood lasted 1 day, not 0
                    "Duration (d)": (end - start).days + 1,
                    "District": district,
                    "Source": f"EM-DAT {row['DisNo.']}",
                    "DataCards": 1,
                    "Source Dataset": "EM-DAT",
                }
            )

    extracted = pd.DataFrame(records)
    extracted["Serial"] = range(EMDAT_SERIAL_BASE, EMDAT_SERIAL_BASE + len(extracted))
    stats = {"rows_scanned": len(emdat), "events_matched": matched_events, "skipped": skipped}
    return extracted, stats


emdat, stats = extract_emdat_records(EMDAT_PATH, MOUNT_ELGON_DISTRICTS)

print(f"{stats['events_matched']} of {stats['rows_scanned']} EM-DAT rows name a Mount Elgon district")
print(f"expanded to {len(emdat)} district-level records")
if stats["skipped"]:
    print(f"skipped for incomplete start/end dates: {stats['skipped']}")

print("\nrecords per district:")
print(emdat["District"].value_counts().to_string())
print("\nduration span (inclusive):", emdat["Duration (d)"].min(), "-", emdat["Duration (d)"].max())

17 of 39 EM-DAT rows name a Mount Elgon district
expanded to 49 district-level records

records per district:
District
MBALE        12
SIRONKO       8
KAPCHORWA     6
BULAMBULI     5
BUTALEJA      5
BUDUDA        4
KWEEN         4
MANAFWA       3
BUKWO         2

duration span (inclusive): 1 - 78


In [206]:
# Nothing should merge silently: check for date+district pairs already present
desinventar_keys = set(df[DATE_COLUMN].astype(str) + "|" + df["District"].str.upper().str.strip())
emdat_keys = emdat[DATE_COLUMN] + "|" + emdat["District"]
overlap = emdat_keys[emdat_keys.isin(desinventar_keys)]

print(f"EM-DAT records landing on a date+district already in DesInventar: {len(overlap)}")
print(f"EM-DAT years: {sorted(set(emdat[DATE_COLUMN].str[:4]))}")
print(f"DesInventar years: {df[DATE_COLUMN].str[:4].min()} - {df[DATE_COLUMN].str[:4].max()}")

df["Source Dataset"] = "DesInventar"
df = pd.concat([df, emdat], ignore_index=True)

print(f"\ncombined frame: {len(df)} rows")
print(df["Source Dataset"].value_counts().to_string())

EM-DAT records landing on a date+district already in DesInventar: 0
EM-DAT years: ['1997', '2002', '2003', '2006', '2007', '2011', '2019', '2020', '2021', '2022', '2023', '2025']
DesInventar years: 1933 - 2018

combined frame: 320 rows
Source Dataset
DesInventar    271
EM-DAT          49


### Findings — merging EM-DAT

**17 of 39 EM-DAT rows name a Mount Elgon district, expanding to 49
district-level records.** No row was skipped for an incomplete date. The combined
frame is **320 rows** (271 DesInventar + 49 EM-DAT).

**No EM-DAT record lands on a date+district pair already in DesInventar** — in
fact not one EM-DAT start date appears anywhere in DesInventar. So the merge adds
records rather than colliding with them, and the duplicate check in section 4 has
no cross-source conflict to resolve.

**The coverage gain is the main benefit.** DesInventar spans 1933–2018; EM-DAT
contributes events in 2019, 2020, 2021, 2022, 2023 and 2025 that DesInventar
does not record at all, along with earlier events in 1997–2011.

Records per district skew towards MBALE (12) and SIRONKO (8), reflecting which
districts EM-DAT names in its admin-unit lists rather than a difference in flood
frequency.

**Two caveats to carry forward:**

1. **EM-DAT rows are national events attributed to districts, not district
   reports.** A single EM-DAT event naming seven districts becomes seven rows
   sharing one start date and one duration. That is a genuine multi-district
   footprint, but it is asserted by EM-DAT's admin-unit list rather than observed
   per district — so section 5's multi-district storm clusters will now be
   dominated by EM-DAT records almost by construction.
2. **The same real flood may appear in both inventories under different dates.**
   Exact-date deduplication cannot detect this, since the two sources disagree on
   when an event started. The 2007 floods, for instance, appear in DesInventar
   dated across several days and in EM-DAT as one event starting 2007/08/15.
   Overlap-based matching would be needed to reconcile them; the current frame
   may therefore double-count a handful of events across sources.

## 3. Date validity

DesInventar pads unknown parts of a date with `0`, so `1933/0/0` is a year-only
record and `2013/10/0` is known only to the month. These are **incomplete rather
than corrupt** — the year is trustworthy, the finer parts were never recorded.

`classify_date` labels each entry's precision so the incomplete ones can be
counted before deciding what to do with them.

In [207]:
def classify_date(value) -> str:
    """Label the precision of a single `Date (YMD)` entry."""
    if pd.isna(value):
        return "missing"

    parts = str(value).split("/")
    if len(parts) != 3 or not all(part.strip().isdigit() for part in parts):
        return "malformed"

    year, month, day = (int(part) for part in parts)
    if month == 0 and day == 0:
        return "year only"
    if month == 0:
        return "day without month"
    if day == 0:
        return "month only"

    # Guards against real calendar errors such as 2013/2/30
    if pd.to_datetime(f"{year}/{month}/{day}", format="%Y/%m/%d", errors="coerce") is pd.NaT:
        return "impossible date"
    return "complete"


def invalid_date_report(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Rows whose date is not a complete calendar date, plus a category count."""
    categories = df[DATE_COLUMN].map(classify_date)
    invalid = df[categories != "complete"].copy()
    invalid["Date Issue"] = categories[categories != "complete"]
    return invalid, categories.value_counts()


invalid, counts = invalid_date_report(df)

print(f"{len(invalid)} of {len(df)} rows have an invalid/incomplete date\n")
print("Breakdown by category:")
print(counts.to_string())

print("\nDistinct invalid values:")
print(invalid[DATE_COLUMN].value_counts().to_string())

48 of 320 rows have an invalid/incomplete date

Breakdown by category:
Date (YMD)
complete             272
month only            42
year only              4
day without month      2

Distinct invalid values:
Date (YMD)
2007/9/0     12
2007/7/0     10
2010/2/0     10
1933/0/0      2
2013/11/0     2
2007/10/0     2
2011/2/0      2
2018/0/4      1
2013/10/0     1
2013/0/19     1
2012/0/0      1
1992/0/0      1
2010/4/0      1
2010/10/0     1
2011/7/0      1


### Findings — dates

**48 of 320 rows (15.0%) have an incomplete date, and every one of them is a
DesInventar row.** All 49 EM-DAT records carry a full start and end date, which is
expected: EM-DAT stores year, month and day in separate numeric fields and the
extraction in section 2 skips any row with a gap.

| Category | Rows | Meaning |
| --- | --- | --- |
| `month only` | 42 | day is `0` — e.g. `2007/9/0`, known to the month |
| `year only` | 4 | month and day both `0` — e.g. `1933/0/0` |
| `day without month` | 2 | month `0` but day set — `2013/0/19`, `2018/0/4` |

No rows are malformed, missing, or impossible — every entry is `YYYY/M/D`
shaped, so zero-padding is the only defect.

Two points that affect interpretation:

- **`2013/0/19` and `2018/0/4` are genuinely odd.** A known day with an unknown
  month is not a precision level; these two are likely data-entry errors.
- **The invalid dates cluster.** `2007/9/0` (12 rows), `2007/7/0` (10) and
  `2010/2/0` (10) account for 32 of the 48. These look like three large flood
  events recorded as many district-level rows sharing one imprecise date — so
  dropping them costs roughly **3 events, not 48 independent ones**. Note EM-DAT
  independently records a 2007 flood (`2007/8/15`, 6 districts), so part of what
  is lost here is recovered from the other source.

### Dropping the incomplete dates

Rows without a full calendar date cannot be aligned to daily rainfall or
soil-moisture observations, which is the purpose of this dataset here, so they
are removed. This is a deliberate trade: the three month-precision events above
are real floods, and they are being discarded for lack of a usable timestamp.

In [208]:
date_categories = df[DATE_COLUMN].map(classify_date)
dropped = int((date_categories != "complete").sum())

df = df[date_categories == "complete"].reset_index(drop=True)

print(f"dropped {dropped} rows with an invalid date")
print(f"df now holds {len(df)} rows ({len(df) / (len(df) + dropped):.1%} of the combined {len(df) + dropped})")
print("\nrows per district after the drop:")
print(df["District"].value_counts().to_string())

dropped 48 rows with an invalid date
df now holds 272 rows (85.0% of the combined 320)

rows per district after the drop:
District
BUTALEJA     61
SIRONKO      53
MANAFWA      38
BULAMBULI    32
BUDUDA       30
MBALE        29
KAPCHORWA    16
BUKWO         7
KWEEN         6


### Findings — effect of the drop

**272 rows retained (85.0%).** The loss falls entirely on DesInventar and is not
geographically uniform:

| District | Combined | After drop | Lost |
| --- | --- | --- | --- |
| BUTALEJA | 73 | 61 | 12 |
| SIRONKO | 66 | 53 | 13 |
| MANAFWA | 45 | 38 | 7 |
| BUDUDA | 39 | 30 | 9 |
| KAPCHORWA | 21 | 16 | 5 |
| BULAMBULI | 33 | 32 | 1 |
| MBALE | 30 | 29 | 1 |
| BUKWO / KWEEN | 7 / 6 | 7 / 6 | 0 |

SIRONKO and BUTALEJA absorb half the loss, while BULAMBULI, MBALE, BUKWO and
KWEEN are barely touched. This mildly biases any per-district event frequency and
is worth stating in the methodology.

## 4. Duplicate records

The dedup key must match the resolution the data will be used at. The target is a
**district-day flood prediction model**, where one training example answers "did
a flood occur in district D on day T". At that resolution two reports of the same
district-day are one example, however differently they were written up, so the
key is **date + district**.

An earlier version of this notebook keyed on date + district + sub-location
(`Subcounty`, `Parish`, `Location`). That was correct for a sub-location-level
analysis but wrong here: it left a single district-day split across up to 8 rows,
which would feed the model the same day many times over and inflate the positive
class.

Deliberately **excluded** from the key:

- **`Comments`, `Description of Cause`, `Source`** — these are the fields that
  vary between re-keyed entries of one event (typos, spacing, the source given as
  `District` / `district file` / `BUTALEJA`). Keying on them re-splits precisely
  the rows that need merging: the `2018/5/4` BUTALEJA district-day has 8 rows
  with 8 distinct `Comments` and 4 distinct `Source` values.
- **`Duration (d)`** — a defensible key component in principle, but it is null in
  76% of rows and, as it happens, never disagrees within a date+district group in
  this dataset, so it cannot discriminate anything here.
- **Sub-location** — see above. The detail is preserved as a count rather than
  thrown away, in `Affected Sub Locations`.

In [209]:
UNSPECIFIED = "~UNSPECIFIED~"

# Columns that hold no analytical payload, so ignoring them when comparing rows
PAYLOAD_EXCLUDED = ["Serial"]


def normalise(series: pd.Series) -> pd.Series:
    """Upper-case and trim a text column so casing/whitespace never split a group.

    Missing values become a sentinel rather than NaN, because NaN != NaN would
    otherwise stop two equally-unspecified values from grouping together.
    """
    return series.astype("string").str.strip().str.upper().fillna(UNSPECIFIED)


def place_key(df: pd.DataFrame) -> pd.Series:
    """Finest recorded sub-location within a district.

    No longer part of the duplicate key, but still needed to count how many
    distinct sub-locations a district-day affected. `Location` is included
    because roughly a third of rows leave `Subcounty` and `Parish` blank and name
    the sub-county in free text instead.
    """
    return (
        normalise(df["Subcounty"])
        + " | "
        + normalise(df["Parish"])
        + " | "
        + normalise(df["Location"])
    )


def duplicate_key(df: pd.DataFrame) -> pd.Series:
    """Identity of a record at the model's resolution: one district, one day."""
    return df[DATE_COLUMN].astype(str) + " | " + normalise(df["District"])

In [210]:
def duplicate_report(df: pd.DataFrame) -> dict[str, object]:
    """Rows reporting the same event on the same district-day."""
    group = duplicate_key(df)
    sizes = group.value_counts()
    repeated = sizes[sizes > 1]

    candidates = df[group.isin(repeated.index)].copy()
    candidates["Duplicate Key"] = group[group.isin(repeated.index)]

    payload = [c for c in df.columns if c not in PAYLOAD_EXCLUDED]

    return {
        "groups": len(repeated),
        "rows": int(repeated.sum()),
        # One row per group is the record to keep; the rest are the excess
        "redundant_rows": int(repeated.sum() - len(repeated)),
        "hard_duplicate_rows": int(df.duplicated(subset=payload, keep=False).sum()),
        "candidates": candidates.sort_values(["Duplicate Key", "Serial"]),
    }


dupes = duplicate_report(df)

print("Same event + date + district")
print(f"  duplicate groups:      {dupes['groups']}")
print(f"  rows involved:         {dupes['rows']}")
print(f"  redundant rows:        {dupes['redundant_rows']}  (dropping these keeps one per district-day)")
print(f"  identical but Serial:  {dupes['hard_duplicate_rows']}")
print(f"  distinct district-days: {duplicate_key(df).nunique()}")

# What the discarded sub-location key would have produced, for comparison
strict = df[DATE_COLUMN].astype(str) + " | " + normalise(df["District"]) + " | " + place_key(df)
print(f"\nkeying on date + district + sub-location instead would leave "
      f"{strict.nunique()} rows rather than {duplicate_key(df).nunique()}")

Same event + date + district
  duplicate groups:      32
  rows involved:         159
  redundant rows:        127  (dropping these keeps one per district-day)
  identical but Serial:  0
  distinct district-days: 145

keying on date + district + sub-location instead would leave 251 rows rather than 145


In [211]:
# Which columns actually disagree inside a district-day group? This decides the
# merge rule for each one.
candidates = dupes["candidates"]
conflicts = {
    column: int((candidates.groupby("Duplicate Key")[column].apply(lambda s: s.dropna().astype(str).nunique()) > 1).sum())
    for column in df.columns
}
print(f"columns with conflicting values inside a district-day (of {dupes['groups']} groups):")
for column, groups in sorted(conflicts.items(), key=lambda kv: -kv[1]):
    if groups:
        print(f"  {column:<24} {groups}")

print("\nthe only conflicting Cause values:")
cause_conflicts = candidates.groupby("Duplicate Key")["Cause"].apply(lambda s: s.dropna().unique())
for key, values in cause_conflicts[cause_conflicts.map(len) > 1].items():
    print(f"  {key:<26} {list(values)}")

print("\nDeaths reported more than once within a district-day:")
tolls = candidates.groupby("Duplicate Key")["Deaths"].agg(["count", "sum"])
print(tolls[tolls["count"] > 1].to_string())

columns with conflicting values inside a district-day (of 32 groups):
  Serial                   32
  Comments                 22
  Code Subcounty           14
  Subcounty                14
  Description of Cause     13
  Location                 11
  Source                   7
  Code Parish              6
  Parish                   6
  Cause                    2
  Deaths                   1

the only conflicting Cause values:
  2017/4/17 | BULAMBULI      ['Windstorm', 'Deforestation']
  2018/5/4 | BUTALEJA        ['Heavy Rain', 'El Niño']

Deaths reported more than once within a district-day:
                     count  sum
Duplicate Key                  
2011/9/16 | SIRONKO      2  7.0


### Findings — duplicates

**32 of the 145 district-days were reported more than once, accounting for 127
redundant rows.**

| Check | Result |
| --- | --- |
| Distinct district-days | 145 |
| District-days with >1 report | 32 |
| Rows involved | 159 |
| Redundant rows | 127 |
| Identical in every column but `Serial` | 0 |
| Rows the sub-location key would have left | 251 |

The last line is the point: the same 272 rows reduce to **145** at district-day
resolution but **251** if sub-location is in the key. The earlier key was leaving
a single district-day split up to 24 ways.

**Every duplicate group is DesInventar-internal.** No group mixes the two
sources, which follows from section 2 finding zero cross-source date+district
collisions — so none of the merge rules below ever has to reconcile a DesInventar
figure against an EM-DAT one.

What actually conflicts inside a district-day, and so needs a merge rule:

| Column | Groups affected |
| --- | --- |
| `Comments` | 22 |
| `Subcounty` / `Code Subcounty` | 14 |
| `Description of Cause` | 13 |
| `Location` | 11 |
| `Source` | 7 |
| `Parish` / `Code Parish` | 6 |
| `Cause` | 2 |
| `Deaths` | 1 |

Two results make the merge safe:

- **`Cause` conflicts in only 2 of 32 groups** — `2017/4/17` BULAMBULI
  (Windstorm vs Deforestation) and `2018/5/4` BUTALEJA (Heavy Rain x6 vs
  El Nino). Only the first is a genuine tie; the second resolves cleanly to
  Heavy Rain on frequency.
- **`Deaths` is reported twice in only 1 group** (`2011/9/16` SIRONKO, 2 and 5),
  so summing to a district toll of 7 affects a single row.

`Duration (d)` does not appear in the conflict list at all — no district-day has
two different reported durations, so taking the maximum is unambiguous here.

### Collapsing to one row per district-day

One row survives per district-day. Each column is merged by a rule chosen from
what the diagnostic above shows actually conflicts:

| Column(s) | Rule | Why |
| --- | --- | --- |
| `Duration (d)` | maximum | Longest reported span, so the flood is labelled active for its full extent. Never actually conflicts in this data. |
| `Deaths`, `DataCards` | sum | Sub-location tolls add up to the district toll. Summing also preserves the grand total, which is checkable. |
| `Cause` | most frequent non-null | Nulls must not outvote a recorded cause. Ties break alphabetically so the result is deterministic. |
| `Comments`, `Description of Cause`, `Source` | longest | The most complete narrative, rather than whichever happened to sort first. |
| `Serial` | lowest | A stable event identifier for the daily expansion in section 5. |
| `Subcounty`, `Parish`, `Location` + codes | **dropped** | Meaningless once a row represents a whole district. Keeping one of eight sub-counties would misrepresent the row, so they are replaced by a count. |
| — | `Affected Sub Locations` | New column: how many distinct sub-locations the district-day affected. A severity proxy for the model, and the only part of the sub-location detail that survives. |

In [212]:
# Merge rules for the columns that conflict within a district-day
NUMERIC_AGGREGATION = {
    "Duration (d)": "max",  # longest reported span for the district-day
    "Deaths": "sum",        # per-sub-location tolls add up to the district toll
    "DataCards": "sum",
}
LONGEST_TEXT = ["Comments", "Description of Cause", "Source"]

# Dropped: a single district-day covers many sub-locations, so naming one of them
# would misrepresent the row. Replaced by `Affected Sub Locations`.
SUB_LOCATION_COLUMNS = ["Subcounty", "Code Subcounty", "Parish", "Code Parish", "Location"]


def collapse_group(group: pd.DataFrame) -> pd.Series:
    """Reduce one district-day to a single row."""
    row = group.iloc[0].copy()
    row["Serial"] = group["Serial"].min()

    for column, how in NUMERIC_AGGREGATION.items():
        # min_count keeps an all-null column null instead of turning it into 0
        row[column] = group[column].max() if how == "max" else group[column].sum(min_count=1)

    for column in LONGEST_TEXT:
        present = group[column].dropna().astype(str)
        if not present.empty:
            row[column] = max(present, key=len)

    causes = group["Cause"].dropna()
    if not causes.empty:
        # Most frequent cause, alphabetical on ties so re-runs agree
        row["Cause"] = min(causes.value_counts().items(), key=lambda kv: (-kv[1], kv[0]))[0]

    row["Affected Sub Locations"] = place_key(group).nunique()
    return row.drop(labels=SUB_LOCATION_COLUMNS)


def collapse_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """One row per district-day; single-report district-days pass through."""
    # Sorting by Serial makes the surviving row deterministic across runs
    ordered = df.assign(_group=duplicate_key(df)).sort_values(["_group", "Serial"])

    collapsed = pd.DataFrame(
        [collapse_group(rows) for _, rows in ordered.groupby("_group", sort=False)]
    ).drop(columns="_group")

    # Rebuilding from row Series loses the original dtypes, so restore the survivors
    retained = {c: t for c, t in df.dtypes.items() if c not in SUB_LOCATION_COLUMNS}
    return collapsed.astype(retained).reset_index(drop=True)


before = len(df)
deaths_before = df["Deaths"].sum()
cards_before = df["DataCards"].sum()

df = collapse_duplicates(df)

print(f"{before} -> {len(df)} rows ({before - len(df)} redundant rows removed)")

print("\ndistrict-days affecting more than one sub-location:")
multi = df[df["Affected Sub Locations"] > 1]
print(multi[[DATE_COLUMN, "District", "Affected Sub Locations", "Duration (d)", "Deaths"]].to_string(index=False))

# Summed columns must preserve their totals, and the key must now be unique
print(f"\nDeaths total preserved:      {df['Deaths'].sum() == deaths_before} ({df['Deaths'].sum():.0f})")
print(f"DataCards total preserved:   {df['DataCards'].sum() == cards_before} ({df['DataCards'].sum()})")
print(f"remaining duplicate groups:  {duplicate_report(df)['groups']}")
print(f"one row per district-day:    {duplicate_key(df).is_unique}")

272 -> 145 rows (127 redundant rows removed)

district-days affecting more than one sub-location:
Date (YMD)  District  Affected Sub Locations  Duration (d)  Deaths
 2005/5/16     MBALE                       2         150.0     NaN
2006/11/28  BUTALEJA                       2           NaN     NaN
 2007/10/1    BUDUDA                       8           NaN     2.0
  2007/8/8   SIRONKO                       2           NaN     NaN
  2009/4/6     MBALE                       2           2.0     NaN
 2010/10/4 BULAMBULI                       8          30.0     NaN
 2010/2/20  BUTALEJA                       2           NaN     NaN
 2010/3/10   MANAFWA                      24          14.0     NaN
 2010/3/25     BUKWO                       4           NaN     NaN
 2010/5/14   SIRONKO                       5           NaN     NaN
 2011/9/15 BULAMBULI                       7           3.0     NaN
 2011/9/15   SIRONKO                       7           NaN     2.0
 2011/9/16   SIRONKO           

### Findings — after collapsing

**272 -> 145 rows**, one per district-day. The re-check reports 0 duplicate groups
and the key is unique. `Deaths` still totals 34 and `DataCards` still totals 272,
which is the check that summing moved values rather than losing them.

**28 of the 145 district-days affected more than one sub-location**, up to 24 for
`2010/3/10` MANAFWA. That count is now the `Affected Sub Locations` column — a
plausible severity feature, since a flood reaching 24 sub-counties is a larger
event than one reaching one, and it is the only trace of the sub-location detail
that survives the merge.

Note the count is distinct *sub-locations*, not reports merged: `2018/5/4`
BUTALEJA was built from 8 reports but only 3 distinct sub-locations, because 6 of
those reports named the same sub-county. **All 49 EM-DAT rows have a count of 1**,
since EM-DAT does not record below district level — so this feature is only
informative for DesInventar-sourced rows, and `Source Dataset` should be used
alongside it rather than treating a 1 as evidence of a small flood.

**Trade-off accepted here:** `Subcounty`, `Parish`, `Location` and their code
columns are dropped. Any later sub-district analysis must go back to the raw
file — this frame cannot answer sub-county questions.

## 5. Multi-district storms

One weather system hitting several neighbouring districts is recorded as
separate district rows sharing a date, so candidate storms are found by grouping
on date (+ `Cause`) and keeping the groups that span more than one district.

`Cause` does little separating work here — it is `Heavy Rain` in the large
majority of rows with a number of nulls — so the date does most of the grouping.

In [213]:
def storm_clusters(df: pd.DataFrame, use_cause: bool = True) -> pd.DataFrame:
    """Group rows into candidate storms and keep those touching >1 district."""
    key = df[DATE_COLUMN].astype(str)
    if use_cause:
        key = key + " | " + normalise(df["Cause"])

    grouped = pd.DataFrame({"key": key, "district": normalise(df["District"])}).groupby("key")
    clusters = grouped["district"].agg(
        districts="nunique",
        rows="size",
        district_names=lambda s: ", ".join(sorted(set(s))),
    )
    return clusters[clusters["districts"] > 1].sort_values("districts", ascending=False)


# Every remaining row has a complete date, so same-date grouping means same *day*
# rather than, as it would for entries like "2007/9/0", merely the same month.
for key_label, use_cause in [("date + cause", True), ("date only", False)]:
    clusters = storm_clusters(df, use_cause=use_cause)
    print(
        f"{key_label:<13} {len(clusters):>2} multi-district clusters, "
        f"{clusters['rows'].sum():>3} rows, max {clusters['districts'].max()} districts"
    )

print("\nMulti-district clusters on date + cause:")
print(storm_clusters(df).to_string())

date + cause  18 multi-district clusters,  57 rows, max 8 districts
date only     19 multi-district clusters,  59 rows, max 8 districts

Multi-district clusters on date + cause:
                            districts  rows                                                       district_names
key                                                                                                             
2019/12/18 | ~UNSPECIFIED~          8     8  BUDUDA, BUKWO, BULAMBULI, KAPCHORWA, KWEEN, MANAFWA, MBALE, SIRONKO
2022/7/30 | ~UNSPECIFIED~           6     6               BUDUDA, BULAMBULI, BUTALEJA, KAPCHORWA, MBALE, SIRONKO
2007/8/15 | ~UNSPECIFIED~           6     6                    BUDUDA, BUKWO, KAPCHORWA, MANAFWA, MBALE, SIRONKO
2011/8/20 | ~UNSPECIFIED~           5     5                           BULAMBULI, BUTALEJA, KWEEN, MBALE, SIRONKO
2021/9/17 | ~UNSPECIFIED~           4     4                                     KAPCHORWA, KWEEN, MBALE, SIRONKO
2002/4/26 | ~UNSPECIFIED~      

### Findings — storms

**18 multi-district clusters on date + cause** (19 on date alone), with a maximum
footprint of **8 districts** — every district in the study area.

**This section is now dominated by EM-DAT, almost by construction.** The seven
largest clusters are all EM-DAT records, identifiable by their
`~UNSPECIFIED~` cause, since EM-DAT has no equivalent of DesInventar's `Cause`
column:

| Date | Districts | Source |
| --- | --- | --- |
| 2019/12/18 | 8 | EM-DAT |
| 2022/7/30 | 6 | EM-DAT |
| 2007/8/15 | 6 | EM-DAT |
| 2011/8/20 | 5 | EM-DAT |
| 2021/9/17 | 4 | EM-DAT |
| 2002/4/26, 2025/8/17 | 3 | EM-DAT |
| 11 further clusters | 2 | mixed |

That is expected rather than a defect: an EM-DAT row *is* a national event
attributed to a list of districts, so exploding it produces a perfect
multi-district cluster every time. The DesInventar-only clusters remain what they
were — 2-district neighbour pairs (BUTALEJA–MANAFWA, KAPCHORWA–SIRONKO,
BULAMBULI–SIRONKO, MBALE–SIRONKO).

**The two sources are therefore measuring different things here.** A DesInventar
cluster is independent evidence that neighbouring districts reported flooding on
the same day; an EM-DAT cluster is one organisation's judgement about which
districts one event touched. They should not be pooled into a single "storm
footprint" statistic without saying which is which.

For DesInventar alone, the date cleaning in section 3 mattered a great deal: on
the raw data this check found clusters spanning 4 districts, but the 4-district
cluster was `2007/7/0`, meaning "some time in July 2007" — evidence of a shared
*month*, not a shared storm.

**Caveat.** Clustering uses the reported start date only, so a storm spanning a
date boundary will not cluster under exact-date equality. A proper footprint
would need an interval-overlap test using `Date + Duration`; the counts above are
a lower bound. This now matters more than before, because EM-DAT durations run to
78 days and its start dates disagree with DesInventar's for the same real event.

## 6. Expanding events to their full day span

Each row so far is one district-day *report*, dated on the day the flood began. A
flood with `Duration (d) = 14` in fact affected its district on 14 consecutive
days, and rainfall or soil-moisture observations exist for every one of those
days. To join against daily data, each event is expanded into one row per day it
was active.

Conventions used, and why:

- **`Duration (d) = N` spans N days**, from the reported date through
  `date + (N - 1)` inclusive — DesInventar defines the field as how long the
  event lasted, so a 1-day event must not become two rows.
- **A missing duration counts as 1 day.** 82 of the 145 events (57%) leave the
  field blank — all of them DesInventar rows, since every EM-DAT record has a
  computed duration. Treating those as single-day events keeps every event in the
  output rather than discarding most of the inventory.
- **Durations are capped per source** — 21 days for DesInventar, 2 days for
  EM-DAT. See the two subsections below.

### Why cap DesInventar at 21 days

Najibi & Devineni (2018) classify global flood events by duration into three
bands — **short (1–7 days), moderate (8–20 days), and long (21 days and above)**
— derived from the Dartmouth Flood Observatory global inventory.

> Najibi, N. and Devineni, N.: *Recent trends in the frequency and duration of
> global floods*, Earth Syst. Dynam., 9, 757–783, 2018.
> https://esd.copernicus.org/articles/9/757/2018/

Three records in this dataset report durations far above that threshold — one of
150 days and two of 30 — and taken literally they would supply 55% of all daily
rows from 3% of the events. A 150-day continuous flood is not plausible as a
single event; it reads as a whole-season or whole-episode summary entered against
one start date.

Capping at 21 rather than some rounder number is deliberate: **21 is the exact
floor of the "long" class, so the cap is class-preserving.** Every event keeps the
same short/moderate/long classification it had under its reported duration, and
nothing is reclassified by the truncation. The cap limits how much a single
over-long record can dominate the daily frame while leaving the duration
*category* — which is what the literature actually supports — untouched.

**To be clear about the citation:** Najibi & Devineni classify durations, they do
not truncate them. The 21-day cap is a modelling decision taken here, informed by
their taxonomy; it is not a method drawn from the paper. The reported value is
preserved in `Duration (d)` so the cap is reversible, and the classification is
recorded in `Duration Class` computed from the *reported* duration.

> The expansion deliberately produces rows that repeat `Serial`, `District` and
> the narrative fields. These are **not** duplicates: each row is a distinct
> *(event, day)* observation, uniquely identified by `Serial` + `Day Index`.

### Why cap EM-DAT at 2 days

The 21-day cap is the wrong instrument for EM-DAT, and leaving both sources on the
same cap produced a training frame with the balance backwards.

**EM-DAT was a third of the events but three quarters of the rows.** Before this
change: 49 of 145 events (34%) were EM-DAT, yet they supplied 544 of 734 daily
rows (74%). So exactly the labels that are *asserted rather than evidenced* — see
section 2 — would have dominated the model 3:1 over the district-level reports
that actually observe a flood.

That happened because **an EM-DAT duration is the envelope of a national episode,
not the length of a district's flood.** The `2007/8/15` record reports 78 days and
names 6 districts. Expanded at the 21-day cap it manufactured 21 × 6 = **126
district-flood-days** out of one row that evidences a few days per district at
best. Nothing in the record says Bukwo was flooded for three weeks; it says a
national flood episode ran 78 days and Bukwo was among the districts EM-DAT
listed for it.

Capping EM-DAT at **2 days** keeps what the record genuinely supports — that a
flood occurred in these districts around this date — without inflating one
national episode into hundreds of district-day observations. The reported value
stays in `Duration (d)`, and `Duration Class` is still computed from it, so the
`2007/8/15` record is still labelled `long` even though it now spans 2 rows per
district.

**Note this cap is deliberately not band-preserving.** Unlike the 21-day
DesInventar cap, which sits exactly on the "long" boundary, capping a 78-day
episode at 2 days moves it from `long` to `short` on span alone. That is accepted:
the point is to stop treating an episode envelope as a district observation, and
`Duration Class` retains the reported band for any model that wants it. The
notebook checks and reports band preservation per source rather than asserting it
for both.

In [214]:
# Najibi & Devineni (2018) duration bands: short 1-7, moderate 8-20, long 21+
DURATION_BANDS = [(7, "short"), (20, "moderate")]

# DesInventar: 21 is the floor of the "long" band, so capping there cannot move an
# event between bands. EM-DAT: 2 days, because its duration is the envelope of a
# national episode rather than a district's flood - see the discussion above.
MAX_EVENT_DAYS_BY_SOURCE = {"DesInventar": 21, "EM-DAT": 2}


def classify_duration(value) -> str:
    """Band a reported duration, keeping unrecorded durations distinguishable.

    Nulls are `unknown` rather than `short`: those events never recorded a
    duration, which is a reporting artefact and should not be conflated with the
    events genuinely reported as lasting one day.
    """
    if pd.isna(value):
        return "unknown"
    for upper, label in DURATION_BANDS:
        if value <= upper:
            return label
    return "long"


def expand_to_daily(df: pd.DataFrame) -> pd.DataFrame:
    """One row per day each event was active, capped per source.

    Adds `Duration Class` (from the reported duration, uncapped), `Observation
    Date` (the calendar day, as a datetime for joining to daily
    rainfall/soil-moisture series), `Day Index` (1-based position within the
    event) and `Event Days` (the capped span actually used).
    """
    caps = df["Source Dataset"].map(MAX_EVENT_DAYS_BY_SOURCE)
    days = df["Duration (d)"].fillna(1).clip(lower=1).combine(caps, min).astype(int)

    base = df.reset_index(drop=True).assign(
        **{
            "Duration Class": df["Duration (d)"].map(classify_duration).to_numpy(),
            "Event Days": days.to_numpy(),
        }
    )

    expanded = base.loc[base.index.repeat(base["Event Days"])].copy()
    # cumcount runs within each original row, so it numbers the days of one event
    expanded["Day Index"] = expanded.groupby(level=0).cumcount() + 1
    expanded["Observation Date"] = pd.to_datetime(
        expanded[DATE_COLUMN], format="%Y/%m/%d"
    ) + pd.to_timedelta(expanded["Day Index"] - 1, unit="D")

    return expanded.reset_index(drop=True)


events = len(df)
uncapped = int(df["Duration (d)"].fillna(1).clip(lower=1).sum())
shortened = df[df["Duration (d)"] > df["Source Dataset"].map(MAX_EVENT_DAYS_BY_SOURCE)]

df = expand_to_daily(df)

print(df[["Date (YMD)","Duration (d)","District","Source Dataset"]].sort_values("Date (YMD)"))

print(f"expanded {events} events -> {len(df)} rows")
print(f"total rows in df: {len(df)}")
print(f"\nuncapped this would be {uncapped} rows; the per-source caps remove {uncapped - len(df)}")
print(f"caps applied: {MAX_EVENT_DAYS_BY_SOURCE}")
print(f"{len(shortened)} events were shortened "
      f"({shortened['Source Dataset'].value_counts().to_dict()})")

print("\nevents and rows by source:")
by_source = df.groupby("Source Dataset").agg(events=("Serial", "nunique"), rows=("Serial", "size"))
by_source["pct_rows"] = (100 * by_source["rows"] / len(df)).round(1)
print(by_source.to_string())

print(f"\ndistinct calendar days covered: {df['Observation Date'].nunique()}")
print(f"date range: {df['Observation Date'].min():%Y-%m-%d} -> {df['Observation Date'].max():%Y-%m-%d}")

# Serial + Day Index must identify a row uniquely, or the expansion double-counted
print(f"Serial + Day Index unique: {df.groupby(['Serial', 'Day Index']).ngroups == len(df)}")
print(f"Event Days sums to row count: {int(df.groupby('Serial')['Event Days'].first().sum()) == len(df)}")

# Band preservation only holds for the 21-day cap, which sits exactly on the
# "long" boundary. The 2-day EM-DAT cap deliberately breaks it, so it is checked
# and reported separately rather than asserted across both sources.
for source, cap in MAX_EVENT_DAYS_BY_SOURCE.items():
    recorded = df[(df["Source Dataset"] == source) & df["Duration (d)"].notna()].drop_duplicates("Serial")
    preserved = (recorded["Duration (d)"].map(classify_duration) == recorded["Event Days"].map(classify_duration)).all()
    print(f"{source:<12} cap={cap:>2}d  band preserved for its {len(recorded)} dated events: {preserved}")

     Date (YMD)  Duration (d)   District Source Dataset
0      1991/9/2           NaN      MBALE    DesInventar
1      1994/5/4           NaN  KAPCHORWA    DesInventar
2    1997/11/14          15.0      MBALE         EM-DAT
3    1997/11/14          15.0      MBALE         EM-DAT
4      1998/3/8           NaN      MBALE    DesInventar
5     2001/10/1           NaN     BUDUDA    DesInventar
6     2002/4/26          33.0  KAPCHORWA         EM-DAT
7     2002/4/26          33.0  KAPCHORWA         EM-DAT
8     2002/4/26          33.0      MBALE         EM-DAT
9     2002/4/26          33.0      MBALE         EM-DAT
10    2002/4/26          33.0    SIRONKO         EM-DAT
11    2002/4/26          33.0    SIRONKO         EM-DAT
12     2002/6/1           NaN    SIRONKO    DesInventar
13    2003/4/21          45.0      MBALE         EM-DAT
14    2003/4/21          45.0      MBALE         EM-DAT
15     2003/7/1           3.0      MBALE         EM-DAT
16     2003/7/1           3.0      MBALE        

In [215]:
print("events and rows per duration band:")
by_band = df.groupby("Duration Class").agg(events=("Serial", "nunique"), rows=("Serial", "size"))
print(by_band.reindex(["unknown", "short", "moderate", "long"]).dropna(how="all").to_string())

print("\nrows contributed by each reported duration:")
contribution = df.groupby(df["Duration (d)"].fillna(0)).agg(events=("Serial", "nunique"), rows=("Serial", "size"))
contribution.index = contribution.index.map(lambda d: "not recorded" if d == 0 else f"{int(d)} days")
print(contribution.to_string())

print("\nrows per district, before vs after the expansion:")
comparison = pd.DataFrame(
    {
        "events": df.groupby("District")["Serial"].nunique(),
        "day_rows": df["District"].value_counts(),
    }
).sort_values("day_rows", ascending=False)
print(comparison.to_string())

# Distinct events overlapping on the same district-day: expected, not duplication
district_days = df.groupby(["District", "Observation Date"])["Serial"].nunique()
print(f"\ndistrict-day combinations:                 {len(district_days)}")
print(f"...drawing on more than one event record:  {int((district_days > 1).sum())}")

print("\nexample — one event unrolled across its span:")
longest = df.loc[df["Event Days"].idxmax(), "Serial"]
sample = df[df["Serial"] == longest]
print(sample[["Serial", DATE_COLUMN, "Duration (d)", "Duration Class", "Event Days", "Day Index", "Observation Date"]].head(4).to_string(index=False))
print(f"... {len(sample)} rows in total for Serial {longest}")

events and rows per duration band:
                events  rows
Duration Class              
unknown             82    82
short               34    64
moderate            12    42
long                17    91

rows contributed by each reported duration:
              events  rows
Duration (d)              
not recorded      82    82
1 days            13    13
2 days             2     4
3 days             5    11
5 days             2     4
6 days             6    20
7 days             6    12
8 days             1     8
9 days             2     4
10 days            2     4
14 days            1    14
15 days            1     2
20 days            5    10
23 days            2     4
25 days            1     2
30 days            2    42
33 days            3     6
45 days            1     2
58 days            1     2
78 days            6    12
150 days           1    21

rows per district, before vs after the expansion:
           events  day_rows
District                   
MBALE          26 

### Findings — daily expansion

**145 district-day events expand to 279 daily rows**, covering 191 distinct
calendar days between 1991-09-02 and 2025-08-18. `Serial` + `Day Index` is unique
across all 279 rows and the per-event spans sum back to the row count, so no day
is double-counted.

**The per-source caps remove 1,049 rows** — uncapped the frame would be 1,328
rows. 43 events were shortened: **40 EM-DAT** (by the 2-day cap) and 3 DesInventar
(by the 21-day cap).

**The source balance is now the right way round**, which was the whole point of
the per-source cap:

| Source | Events | Rows | Share of rows |
| --- | --- | --- | --- |
| DesInventar | 96 (66%) | 190 | **68.1%** |
| EM-DAT | 49 (34%) | 89 | **31.9%** |

Row share now tracks event share almost exactly (68/32 against 66/34). Before the
change EM-DAT held 74% of rows on 34% of events; the district-level reports that
actually observe flooding now carry the model.

Rows by duration band, computed from the *reported* duration:

| Band | Events | Rows |
| --- | --- | --- |
| unknown (not recorded) | 82 | 82 |
| short (1–7 days) | 34 | 64 |
| moderate (8–20 days) | 12 | 42 |
| long (21+ days) | 17 | 91 |

Band preservation holds for DesInventar's 14 dated events and, as intended, does
not hold for EM-DAT's 49 — the 2-day cap moves long episodes to a 2-day span by
design.

Rows per district:

| District | Events | Day rows |
| --- | --- | --- |
| MBALE | 26 | 85 |
| BULAMBULI | 14 | 40 |
| BUTALEJA | 29 | 35 |
| SIRONKO | 26 | 33 |
| BUDUDA | 15 | 28 |
| MANAFWA | 10 | 25 |
| KAPCHORWA | 16 | 21 |
| KWEEN | 6 | 8 |
| BUKWO | 3 | 4 |

MBALE is still the largest at 85 rows, and 21 of those come from the single
150-day DesInventar record capped to 21 days — so the residual imbalance is now
driven by that one DesInventar outlier rather than by EM-DAT episode envelopes.
The smallest districts (BUKWO 4 rows, KWEEN 8) are thin, which is the honest
consequence of not inflating spans: these districts genuinely have few observed
flood days in either inventory.

Only 7 of 272 district-day combinations draw on more than one event record.

## 7. Result

`df` now holds **279 daily rows** describing 145 district-day flood events, drawn
from 271 DesInventar records and 49 EM-DAT district-records:

| Stage | Rows | Change |
| --- | --- | --- |
| Loaded (DesInventar) | 271 | — |
| Merged EM-DAT records | 320 | +49 district-records from 17 events |
| Complete dates only | 272 | −48 incomplete dates (all DesInventar) |
| Collapsed to one row per district-day | 145 | −127 redundant reports |
| Expanded to day spans (21d / 2d by source) | 279 | +134 event-days |

Row share by source is DesInventar 190 (68.1%) and EM-DAT 89 (31.9%), tracking
the 66/34 split in event counts.

Each row is one *(district-day event, day)* pair, uniquely keyed by `Serial` +
`Day Index`, carrying:

- `Observation Date` — datetime, ready to join against daily rainfall and
  soil-moisture series
- `Source Dataset` — `DesInventar` or `EM-DAT`
- `Duration Class` — Najibi & Devineni band from the reported, uncapped duration
  (`unknown` / `short` / `moderate` / `long`)
- `Event Days` — the capped span actually expanded
- `Duration (d)` — the duration as reported, so both caps are reversible
- `Affected Sub Locations` — sub-location count; always 1 for EM-DAT rows

Known limitations to carry into the analysis:

1. **The truncated days are dropped, not marked unknown.** Capping EM-DAT at 2
   days discards 902 days that fell inside a reported episode, and the DesInventar
   cap discards 147 more. Those days are not evidence of dry conditions — they are
   simply unobserved. If negative district-days are sampled from the calendar for
   training, days inside a reported episode window should be excluded rather than
   treated as flood-free, or the model will be taught that real flood episodes
   were dry.
2. **The same real flood may appear in both inventories under different dates.**
   Exact-date deduplication cannot detect this, so a handful of events are likely
   double-counted across sources — the 2007 floods appear in DesInventar across
   several imprecise dates and in EM-DAT as one event starting `2007/8/15`.
   Reconciling them needs interval-overlap matching, not equality.
3. **EM-DAT rows are national events attributed to districts, not per-district
   observations.** They assert a multi-district footprint rather than evidencing
   it, which is why the 2-day cap exists and why section 5's largest storm
   clusters are all EM-DAT. Do not pool EM-DAT and DesInventar clusters into one
   footprint statistic.
4. **Provenance is partly inferable from the features.** `Cause` is null and
   `Affected Sub Locations` is 1 for every EM-DAT row, and EM-DAT spans are now
   at most 2 days, so those columns together identify the source. Feeding them to
   a model risks it learning which inventory a row came from rather than flood
   conditions; evaluate per source to check.
5. EM-DAT's `Total Deaths` and `Total Affected` are omitted, being event-level
   totals that cannot be attributed per district. `Deaths` is therefore null for
   all EM-DAT rows and present in only 17 DesInventar rows, so impact severity
   cannot be modelled from this frame.
6. Three DesInventar flood events (`2007/7/0`, `2007/9/0`, `2010/2/0`) were
   dropped for month-only dates; date-filter loss is uneven across districts,
   falling mostly on SIRONKO and BUTALEJA.
7. **57% of district-day events (82 of 145) never recorded a duration** and count
   as a single day. Day-row counts therefore partly measure reporting
   completeness rather than flood extent. Check label balance per district before
   training — BUKWO has 4 rows and KWEEN 8.
8. Both caps are modelling decisions, not source facts. The 21-day DesInventar cap
   is informed by Najibi & Devineni's duration bands and is band-preserving; the
   2-day EM-DAT cap is deliberately not, and is justified by the unit-of-record
   mismatch rather than by any published threshold.
9. `Subcounty`, `Parish` and `Location` are dropped by the district-day merge.
   Only the `Affected Sub Locations` count survives, and it is uninformative for
   EM-DAT rows; sub-district questions need the raw files.
10. `Cause` was resolved by majority vote in the 2 district-days where reports
    disagreed, so `2017/4/17` BULAMBULI (Windstorm vs Deforestation) carries an
    arbitrary alphabetical tie-break. EM-DAT rows have no `Cause` at all.